In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 Quantum Key Distribution — No Attacker

This notebook simulates the BB84 protocol between **Alice** and **Bob** with no eavesdropper.

## Protocol overview

1. **Alice** generates random bits and random bases using quantum measurement, then encodes each bit as a qubit.
2. **Bob** generates random measurement bases using quantum measurement, then measures each qubit.
3. **Sifting**: Alice and Bob publicly compare *bases* (not bits). They keep only the bits where they used the same basis — this is the *sifted key*.
4. **Error checking**: They sacrifice a small portion of the sifted key to compute the *Quantum Bit Error Rate (QBER)*. A low QBER confirms no eavesdropping.

### Encoding convention
| Basis | Bit 0 | Bit 1 |
|-------|-------|-------|
| Rectilinear `+` | `|0⟩` | `|1⟩` |
| Diagonal `×`   | `|+⟩ = H|0⟩` | `|−⟩ = H|1⟩` |

## Quantum random bit generation

All random choices are made by measuring $\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, the equal superposition state produced by applying a Hadamard gate to $|0\rangle$. This gives a truly random 0 or 1 with equal probability, grounded in quantum mechanics — **no classical `random` module is used**.

In [2]:
# ── Shared utility ──────────────────────────────────────────────────────────
# Used by all agents to make random choices via quantum measurement.

simulator = BasicSimulator()

def quantum_random_bit():
    """
    Return a single random classical bit (0 or 1) by:
      1. Preparing |0⟩
      2. Applying H to get 1/√2 (|0⟩ + |1⟩)
      3. Measuring — collapses to |0⟩ or |1⟩ with equal probability
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0⟩ → 1/√2 (|0⟩ + |1⟩)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    """Return a list of n quantum-random bits."""
    return [quantum_random_bit() for _ in range(n)]

## Alice

Alice randomly generates:
- **`alice_bits`** — the secret bits she wants to encode
- **`alice_bases`** — the basis she uses for each qubit (`0` = rectilinear `+`, `1` = diagonal `×`)

She then prepares one qubit per bit and "sends" them to Bob (represented here as a list of `QuantumCircuit` objects).

In [3]:
# ── ALICE ────────────────────────────────────────────────────────────────────

N = 100  # number of qubits to send

# Alice generates her secret random bits and random bases — all via quantum measurement
alice_bits  = quantum_random_bits(N)   # the bits to encode
alice_bases = quantum_random_bits(N)   # 0 = rectilinear (+), 1 = diagonal (×)

def alice_encode(bit, basis):
    """
    Prepare a single qubit encoding `bit` in the given `basis`.

    Basis 0 (rectilinear +):
        bit=0 → |0⟩   (no gate needed)
        bit=1 → |1⟩   (X gate)
    Basis 1 (diagonal ×):
        bit=0 → |+⟩   (H gate)
        bit=1 → |−⟩   (X then H)
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)   # flip to |1⟩ before basis encoding
    if basis == 1:
        qc.h(0)   # rotate into diagonal basis
    return qc

# Alice prepares and "transmits" the qubits
transmitted_qubits = [alice_encode(b, basis) for b, basis in zip(alice_bits, alice_bases)]

print("=" * 60)
print("ALICE")
print("=" * 60)
print(f"Alice's bits  (first 20): {alice_bits[:20]}")
print(f"Alice's bases (first 20): {alice_bases[:20]}")
print(f"  (basis key: 0=rectilinear +, 1=diagonal ×)")
print(f"\nAlice has prepared and transmitted {N} qubits.")

ALICE
Alice's bits  (first 20): [0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]
Alice's bases (first 20): [1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]
  (basis key: 0=rectilinear +, 1=diagonal ×)

Alice has prepared and transmitted 100 qubits.


## Bob

Bob randomly chooses a measurement basis for each qubit he receives. He then measures each qubit in his chosen basis:
- In the **rectilinear** basis: measure directly in the computational basis.
- In the **diagonal** basis: apply H first to rotate back, then measure.

In [4]:
# ── BOB ──────────────────────────────────────────────────────────────────────

# Bob independently chooses random measurement bases — via quantum measurement
bob_bases = quantum_random_bits(N)   # 0 = rectilinear (+), 1 = diagonal (×)

def bob_measure(qubit_circuit, basis):
    """
    Measure the received qubit in the given basis.

    If basis == 1 (diagonal ×), apply H first to rotate the diagonal
    basis back to the computational basis before measuring.
    """
    qc = qubit_circuit.copy()
    if basis == 1:
        qc.h(0)   # rotate diagonal basis back to computational basis
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

# Bob measures each received qubit
bob_results = [bob_measure(qc, basis) for qc, basis in zip(transmitted_qubits, bob_bases)]

print("=" * 60)
print("BOB")
print("=" * 60)
print(f"Bob's bases   (first 20): {bob_bases[:20]}")
print(f"Bob's results (first 20): {bob_results[:20]}")
print(f"  (basis key: 0=rectilinear +, 1=diagonal ×)")

BOB
Bob's bases   (first 20): [1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1]
Bob's results (first 20): [0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1]
  (basis key: 0=rectilinear +, 1=diagonal ×)


## Sifting

Alice and Bob announce their **bases** over a public channel. They keep only the positions where their bases matched — these form the **sifted key**. On average, about 50% of bits are retained.

In [5]:
# ── SIFTING (public classical channel) ───────────────────────────────────────
# Alice and Bob publicly compare bases and discard mismatches.
# Only the BASES are revealed — the bit values remain secret.

alice_sifted = []
bob_sifted   = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:   # bases matched → keep this bit
        alice_sifted.append(alice_bits[i])
        bob_sifted.append(bob_results[i])

sifted_len = len(alice_sifted)

print("=" * 60)
print("SIFTING")
print("=" * 60)
print(f"Original qubits sent : {N}")
print(f"Matching bases       : {sifted_len}  ({100*sifted_len/N:.1f}%)")
print(f"\nAlice sifted key (first 20): {alice_sifted[:20]}")
print(f"Bob   sifted key (first 20): {bob_sifted[:20]}")

SIFTING
Original qubits sent : 100
Matching bases       : 53  (53.0%)

Alice sifted key (first 20): [0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0]
Bob   sifted key (first 20): [0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0]


## Error checking (QBER)

Alice and Bob sacrifice a **check sample** of the sifted key (publicly comparing both bases *and* bit values). They compute the **Quantum Bit Error Rate (QBER)**:

$$\text{QBER} = \frac{\text{number of mismatched check bits}}{\text{total check bits}}$$

- Without an attacker: QBER ≈ 0% (only simulator noise, if any)
- With an intercept-resend attacker: QBER ≈ 25%

A threshold of **15%** is used: above this the protocol aborts.

In [6]:
# ── ERROR CHECKING ────────────────────────────────────────────────────────────
# Alice and Bob publicly compare a subset of their sifted key to estimate QBER.

QBER_THRESHOLD = 0.15   # 15%: above this, assume an attacker is present
CHECK_FRACTION = 0.25   # sacrifice 25% of the sifted key for error checking

check_size = max(1, int(sifted_len * CHECK_FRACTION))

# Use the first `check_size` bits as the check sample
# (in a real implementation these positions would be chosen randomly)
alice_check = alice_sifted[:check_size]
bob_check   = bob_sifted[:check_size]

# The remaining bits form the final secret key
alice_key = alice_sifted[check_size:]
bob_key   = bob_sifted[check_size:]

# Compute QBER
errors = sum(a != b for a, b in zip(alice_check, bob_check))
qber   = errors / check_size if check_size > 0 else 0

print("=" * 60)
print("ERROR CHECKING")
print("=" * 60)
print(f"Check sample size : {check_size} bits")
print(f"Errors found      : {errors}")
print(f"QBER              : {qber:.2%}")
print(f"Threshold         : {QBER_THRESHOLD:.0%}")

ERROR CHECKING
Check sample size : 13 bits
Errors found      : 0
QBER              : 0.00%
Threshold         : 15%


## Result

In [7]:
# ── RESULT ────────────────────────────────────────────────────────────────────

print("=" * 60)
print("RESULT")
print("=" * 60)

if qber > QBER_THRESHOLD:
    print(f"ATTACK DETECTED — QBER {qber:.2%} exceeds threshold {QBER_THRESHOLD:.0%}.")
    print("Protocol aborted. Key is not trusted.")
else:
    print(f"Channel secure — QBER {qber:.2%} is below threshold {QBER_THRESHOLD:.0%}.")
    print(f"\nFinal shared key length : {len(alice_key)} bits")
    print(f"Alice's key (first 20)  : {alice_key[:20]}")
    print(f"Bob's   key (first 20)  : {bob_key[:20]}")
    print(f"Keys match              : {alice_key == bob_key}")

RESULT
Channel secure — QBER 0.00% is below threshold 15%.

Final shared key length : 40 bits
Alice's key (first 20)  : [1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0]
Bob's   key (first 20)  : [1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0]
Keys match              : True
